# 06_adipose_preprocess

Selected source cells for review and reconstruction.

Source: `Adipose_combined_samples_CellSegmentation/Adipose_Combination_2(Sketch harmony).ipynb`

Raw segmented count matrices. Paths to on-disk matrices must be relocated with the RDS.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 1 (one-based); original execution count 1.

In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(BPCells)
  library(Matrix)
  library(hdf5r)
})

Source cell 2 (one-based); original execution count 2.

In [ ]:
library(harmony)

Source cell 4 (one-based); original execution count 4.

In [ ]:
# 1) outs ディレクトリを基準にする
outs <- list(
  Adipose_Aged_1  = "/work/Visium/Visium_session1/Adipose_Aged_1/outs",
  Adipose_Young_1 = "/work/Visium/Visium_session1/Adipose_Young_1/outs",
  Adipose_Aged_2  = "/work/Visium/Visium_session2/Adipose_Aged_2/outs",
  Adipose_Young_2 = "/work/Visium/Visium_session2/Adipose_Young_2/outs"
)
stopifnot(all(dir.exists(unlist(outs))))

Source cell 5 (one-based); original execution count 5.

In [ ]:
# 2) cell segmentation の h5 ファイルパスに変換
h5_seg <- lapply(
  outs,
  function(o) file.path(o, "segmented_outputs/filtered_feature_cell_matrix.h5")
)
stopifnot(all(file.exists(unlist(h5_seg))))  # .h5 ファイルの存在確認

Source cell 6 (one-based); original execution count 6.

In [ ]:
# 3) on-disk matrix を作って保存するフォルダ（Adipose用に新たに作成）
odir <- "/work/Visium/Adipose_combined_samples/on_disk_mats"
dir.create(odir, recursive = TRUE, showWarnings = FALSE)


Source cell 7 (one-based); original execution count 7.

In [ ]:
# 4) on-disk matrices 作成（Adipose segmentation データ用）
mat_list <- lapply(names(h5_seg), function(nm) {
  m <- BPCells::open_matrix_10x_hdf5(path = h5_seg[[nm]], feature_type = "Gene Expression")
  out <- file.path(odir, nm)
  BPCells::write_matrix_dir(mat = m, dir = out, overwrite = TRUE)
  BPCells::open_matrix_dir(out)
})
names(mat_list) <- names(h5_seg)


Source cell 8 (one-based); original execution count 8.

In [ ]:
# 5) Seurat object（on-diskのまま）
obj_list <- lapply(names(mat_list), function(nm) {
  so <- CreateSeuratObject(
    counts = mat_list[[nm]],  # on-disk matrix
    project = nm,
    assay = "RNA",
    min.cells = 0,
    min.features = 0
  )
  so$sample <- nm  # メタデータに sample 情報を追加
  so <- RenameCells(so, add.cell.id = nm)  # セル名にプレフィックス
  so
})
names(obj_list) <- names(mat_list)


Source cell 9 (one-based); original execution count 9.

In [ ]:
# 6) merge（1つのSeuratに）
merge.obj <- Reduce(function(a,b) merge(a,b), obj_list)
merge.obj

Source cell 12 (one-based); original execution count 12.

In [ ]:
merge.obj <- subset(merge.obj, subset = nCount_RNA >= 25)

Source cell 13 (one-based); original execution count 13.

In [ ]:
DefaultAssay(merge.obj) <- "RNA"

merge.obj <- NormalizeData(
  merge.obj,
  scale.factor = median(merge.obj$nCount_RNA),  # medianライブラリサイズで正規化
  verbose = FALSE
)

merge.obj <- FindVariableFeatures(merge.obj, verbose = FALSE)

Source cell 14 (one-based); original execution count 14.

In [ ]:
merge.obj <- SketchData(
  object = merge.obj,
  ncells = 50000,
  method = "LeverageScore",
  sketched.assay = "sketch"
)

Source cell 16 (one-based); original execution count 16.

In [ ]:
# (4) 以降は sketch
DefaultAssay(merge.obj) <- "sketch"

merge.obj <- NormalizeData(merge.obj, verbose = FALSE)
merge.obj <- FindVariableFeatures(merge.obj, verbose = FALSE)
merge.obj <- ScaleData(merge.obj, verbose = FALSE)
merge.obj <- RunPCA(merge.obj, npcs = 50, verbose = FALSE)

Source cell 17 (one-based); original execution count 17.

In [ ]:
# 7) Batch correction by Harmony（RunHarmonyではなく IntegrateLayers）
merge.obj <- IntegrateLayers(
  merge.obj,
  method = HarmonyIntegration,
  orig = "pca",
  new.reduction = "harmony",
  dims = 1:30
)

Source cell 18 (one-based); original execution count 18.

In [ ]:
# UMAP + clustering（harmony次元で）
merge.obj <- RunUMAP(merge.obj, reduction = "harmony", dims = 1:30, reduction.name = "umap")
merge.obj <- FindNeighbors(merge.obj, reduction = "harmony", dims = 1:30)
merge.obj <- FindClusters(merge.obj, resolution = 0.5)

Source cell 22 (one-based); original execution count 22.

In [ ]:
saveRDS(
  merge.obj,
  file = "adipose_mergeobj_preProject_sketchHarmony.rds"
)

Source cell 23 (one-based); original execution count 23.

In [ ]:
# 8) Project（sketch → full へ投影）
merge.obj <- ProjectIntegration(
  object = merge.obj,
  sketched.assay = "sketch",
  assay = "RNA",
  reduction = "harmony"
)

Source cell 25 (one-based); original execution count 25.

In [ ]:
merge.obj <- ProjectData(
  object = merge.obj,
  assay = "RNA",
  full.reduction = "harmony.full",
  sketched.assay = "sketch",
  sketched.reduction = "harmony.full",
  umap.model = "umap",
  dims = 1:30,
  refdata = list(cluster_full = "seurat_clusters")
)

Source cell 31 (one-based); original execution count 31.

In [ ]:
saveRDS(
  merge.obj,
  file = "adipose_mergeobj_preProject_full_sketchHarmony.rds"
)
